# Multi-Document Policy Assistant

**Project 11** — Core RAG Projects

Build a Retrieval-Augmented Generation (RAG) system that can answer questions across
multiple organizational policy documents, providing citations, source chunk display,
and answer confidence scoring.

**Key Skills:** Multi-document RAG, metadata filtering, citation extraction,
confidence scoring, chunked retrieval, vector similarity search.

## Project Overview

Organizations maintain dozens of policy documents — HR handbooks, IT security policies,
travel guidelines, data privacy rules, remote-work agreements, and more. Employees
frequently need to find answers that span multiple documents.

This notebook builds a **multi-document policy assistant** that:
1. Ingests multiple policy documents with rich metadata (department, category, date)
2. Chunks documents while preserving metadata lineage
3. Builds a searchable vector index with metadata filtering
4. Retrieves relevant chunks with citation tracking
5. Scores answer confidence based on retrieval quality
6. Synthesizes grounded answers with explicit source references

**Architecture:** Documents → Chunker → Embedder → Vector Store → Retriever (with metadata filter) → Synthesizer → Answer + Citations + Confidence

## Learning Goals

By the end of this notebook you will understand:
- How multi-document RAG differs from single-document Q&A
- Why metadata filtering matters for precision retrieval
- How to preserve document lineage through the chunking pipeline
- How to extract and display source citations from retrieved chunks
- How to compute retrieval-based confidence scores
- How to build a complete policy assistant pipeline from scratch

## Problem Statement

**Scenario:** A mid-size company has five core policy documents covering HR, IT Security,
Travel & Expense, Data Privacy, and Remote Work. Employees ask questions like:

- *"How many vacation days do I get after 3 years?"*
- *"What is the password rotation policy?"*
- *"Can I expense meals during travel?"*
- *"How long does the company retain my personal data?"*
- *"Am I eligible for remote work?"*

**Goal:** Build a system that retrieves the right chunks from the right documents,
cites sources precisely, and tells the user how confident it is in the answer.

## Why This Project Matters

| Challenge | How RAG Solves It |
|-----------|------------------|
| Employees can't find answers across 100+ page docs | Semantic search finds relevant passages instantly |
| Answers require cross-referencing multiple policies | Multi-document retrieval surfaces all relevant sources |
| Users need to trust the answer | Citations and confidence scores build trust |
| Policies differ by department/date | Metadata filtering ensures the right version is used |
| Full-text search misses semantic meaning | Vector embeddings capture meaning, not just keywords |

## Multi-Document RAG Architecture

### Why multi-document RAG is harder than single-document

Single-document RAG retrieves chunks from one source. Multi-document RAG must:
- **Track provenance**: every chunk must carry metadata about its source document
- **Handle metadata heterogeneity**: different documents have different schemas
- **Support filtered retrieval**: "only search IT policies" or "policies after 2024"
- **Rank across documents**: chunks from different docs must be compared fairly
- **Cite precisely**: answers must reference specific documents and sections

### Metadata filtering explained

Without filtering, a query about "password policy" might retrieve chunks from the
travel policy that mention passwords in passing. Metadata filters let you:
- Filter by **department**: `department == "IT Security"`
- Filter by **category**: `category == "access_control"`
- Filter by **date range**: `effective_date >= "2024-01-01"`
- Combine filters: `department == "IT" AND category == "access_control"`

This dramatically improves precision by narrowing the search space before
semantic similarity kicks in.

## Environment Setup

This notebook uses these libraries:
- **sentence-transformers** — for dense embeddings
- **chromadb** — for vector storage with metadata filtering
- **numpy** — for numerical operations

If ChromaDB or sentence-transformers are unavailable, the notebook falls back to
a TF-IDF + cosine similarity approach with manual metadata filtering.

In [1]:
import os
import json
import hashlib
import re
import textwrap
import warnings
from datetime import datetime
from collections import defaultdict

import numpy as np

warnings.filterwarnings("ignore")

# ── Try to import optional RAG dependencies ──
USE_CHROMA = False
USE_SENTENCE_TRANSFORMERS = False

try:
    import chromadb
    USE_CHROMA = True
    print("[OK] chromadb available")
except ImportError:
    print("[WARN] chromadb not available — using TF-IDF fallback")

try:
    from sentence_transformers import SentenceTransformer
    USE_SENTENCE_TRANSFORMERS = True
    print("[OK] sentence-transformers available")
except ImportError:
    print("[WARN] sentence-transformers not available — using TF-IDF fallback")

# ── Sklearn for TF-IDF fallback ──
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("\nSetup complete.")

[OK] chromadb available


[OK] sentence-transformers available

Setup complete.


## Configuration

All tuneable parameters are set here. Adjust chunk size, overlap, and retrieval
parameters to experiment with different retrieval strategies.

In [2]:
# ── Configuration ──
CHUNK_SIZE = 500          # characters per chunk
CHUNK_OVERLAP = 100       # overlap between consecutive chunks
TOP_K = 5                 # number of chunks to retrieve
CONFIDENCE_THRESHOLD = 0.5  # minimum confidence to consider answer reliable
EMBEDDING_MODEL = "all-MiniLM-L6-v2"  # sentence-transformer model

SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Chunk size: {CHUNK_SIZE}, Overlap: {CHUNK_OVERLAP}, Top-K: {TOP_K}")

Chunk size: 500, Overlap: 100, Top-K: 5


## Policy Documents

We create **five realistic organizational policy documents**, each representing a
different department. Each document includes:
- **Title and department** — for identification and filtering
- **Effective date and version** — for temporal filtering
- **Category tags** — for topic-based filtering
- **Full policy text** — organized into sections

These serve as our document corpus for the RAG pipeline.

In [3]:
# ── Create sample policy documents ──

POLICY_DOCUMENTS = [
    {
        "doc_id": "HR-001",
        "title": "Employee Handbook — Human Resources Policy",
        "department": "Human Resources",
        "category": "employee_benefits",
        "effective_date": "2024-01-15",
        "version": "3.2",
        "text": """
SECTION 1: LEAVE POLICY
Full-time employees are entitled to paid time off (PTO) as follows:
- 0-2 years of service: 15 days per year
- 3-5 years of service: 20 days per year
- 6-10 years of service: 25 days per year
- 10+ years of service: 30 days per year
PTO requests must be submitted at least 2 weeks in advance through the HR portal.
Unused PTO up to 5 days may be carried over to the next calendar year. Any PTO beyond
5 days will be forfeited on December 31st unless approved by department management.

SECTION 2: CODE OF CONDUCT
All employees must maintain professional behavior at all times. This includes:
- Respectful communication with colleagues and clients
- Appropriate business attire as defined by department standards
- Punctuality for meetings and scheduled work hours
- Compliance with all company policies and applicable laws
Violations of the code of conduct will result in progressive discipline:
first written warning, second written warning, final warning, then termination.

SECTION 3: BENEFITS OVERVIEW
The company offers comprehensive benefits including:
- Health insurance (medical, dental, vision) with 80% employer contribution
- 401(k) retirement plan with 4% employer match after 1 year of service
- Life insurance at 2x annual salary
- Employee Assistance Program (EAP) for counseling and support
- Tuition reimbursement up to $5,000 per year for approved programs
- Parental leave: 12 weeks paid for primary caregivers, 4 weeks for secondary
Open enrollment occurs annually in November. Changes outside open enrollment
require a qualifying life event (marriage, birth, adoption, etc.).

SECTION 4: PERFORMANCE REVIEWS
Performance reviews are conducted semi-annually in June and December.
Reviews follow a 360-degree feedback model including self-assessment,
manager evaluation, and peer feedback. Performance ratings use a 5-point scale:
1-Needs Improvement, 2-Below Expectations, 3-Meets Expectations,
4-Exceeds Expectations, 5-Outstanding. Employees rated 4 or above are
eligible for merit increases of 3-8% and bonus consideration.
"""
    },
    {
        "doc_id": "IT-001",
        "title": "Information Security Policy",
        "department": "IT Security",
        "category": "access_control",
        "effective_date": "2024-03-01",
        "version": "5.1",
        "text": """
SECTION 1: PASSWORD POLICY
All system passwords must meet these requirements:
- Minimum 14 characters in length
- Must contain uppercase, lowercase, numbers, and special characters
- Passwords must be changed every 90 days
- Previous 12 passwords cannot be reused
- Multi-factor authentication (MFA) is mandatory for all systems
- Passwords must not contain personal information or dictionary words
Account lockout occurs after 5 failed login attempts. Locked accounts
require IT Service Desk verification to unlock.

SECTION 2: DATA CLASSIFICATION
Company data is classified into four tiers:
- PUBLIC: Marketing materials, published reports (no restrictions)
- INTERNAL: Internal memos, project plans (employees only)
- CONFIDENTIAL: Financial data, HR records, contracts (need-to-know basis)
- RESTRICTED: Trade secrets, encryption keys, executive strategy (named access only)
Each tier has specific handling, storage, and transmission requirements.
Confidential and Restricted data must be encrypted at rest and in transit.
Data classification labels must appear on all documents and emails.

SECTION 3: INCIDENT RESPONSE
Security incidents must be reported within 1 hour of discovery to the
Security Operations Center (SOC) at security@company.com or ext. 9911.
Incident severity levels:
- SEV1 (Critical): Active data breach, ransomware — response within 15 minutes
- SEV2 (High): Suspected compromise, malware detection — response within 1 hour
- SEV3 (Medium): Policy violation, phishing attempt — response within 4 hours
- SEV4 (Low): Minor policy exception, audit finding — response within 24 hours
All SEV1 and SEV2 incidents require immediate escalation to the CISO.

SECTION 4: ACCEPTABLE USE
Company IT resources are provided for business purposes. Limited personal use
is permitted if it does not interfere with work duties or violate any policy.
Prohibited activities include: unauthorized software installation, cryptocurrency
mining, accessing inappropriate content, using company email for personal business,
sharing credentials, and connecting unauthorized devices to the corporate network.
"""
    },
    {
        "doc_id": "FIN-001",
        "title": "Travel and Expense Policy",
        "department": "Finance",
        "category": "expense_management",
        "effective_date": "2024-06-01",
        "version": "2.4",
        "text": """
SECTION 1: TRAVEL BOOKING
All business travel must be pre-approved by the employee's direct manager.
Travel exceeding $2,000 requires VP-level approval. Bookings should be made
through the corporate travel portal (TravelWorks) at least 14 days in advance.
- Air travel: Economy class for flights under 6 hours; business class permitted
  for flights over 6 hours with VP approval
- Hotels: Maximum $250 per night in standard markets; $350 per night in high-cost
  cities (NYC, SF, London, Tokyo). Use preferred hotel partners when available.
- Rental cars: Compact or mid-size only; upgrades require justification.
- Ground transportation: Ride-sharing and taxis permitted; keep all receipts.

SECTION 2: MEAL ALLOWANCES
Meal expenses during business travel are reimbursed up to these daily limits:
- Breakfast: $20
- Lunch: $30
- Dinner: $50
- Total daily maximum: $100
Alcohol is not reimbursable. Team dinners with clients may exceed individual
limits with manager pre-approval. Tips up to 20% are reimbursable.
Receipts required for all meals over $25.

SECTION 3: EXPENSE REPORTING
Expense reports must be submitted within 30 days of travel completion.
Reports submitted after 60 days will not be reimbursed. Required documentation:
- Original itemized receipts for all expenses over $25
- Business purpose justification for each expense
- Manager approval signature
- Client/project code for billing allocation
Reimbursement is processed within 10 business days of approved submission.
Corporate credit card statements must be reconciled monthly.

SECTION 4: NON-TRAVEL EXPENSES
Non-travel business expenses require pre-approval:
- Office supplies under $100: No approval needed
- Software/subscriptions under $500: Manager approval
- Equipment/hardware under $2,000: Director approval
- Any expense over $2,000: VP approval
- Conference/training registration: Manager + HR approval
"""
    },
    {
        "doc_id": "LEGAL-001",
        "title": "Data Privacy and Protection Policy",
        "department": "Legal",
        "category": "data_privacy",
        "effective_date": "2024-02-01",
        "version": "4.0",
        "text": """
SECTION 1: DATA COLLECTION PRINCIPLES
The company collects personal data only for specified, explicit, and legitimate
purposes. Data minimization is enforced — only data necessary for the stated
purpose may be collected. All data collection requires:
- Clear privacy notice provided to data subjects
- Explicit consent for sensitive data categories
- Documented legal basis (consent, contract, legitimate interest, legal obligation)
- Data Protection Impact Assessment (DPIA) for high-risk processing
The Data Protection Officer (DPO) must approve all new data collection initiatives.

SECTION 2: DATA RETENTION
Personal data is retained only as long as necessary for its original purpose
or as required by law. Standard retention periods:
- Employee records: 7 years after termination
- Customer transaction data: 5 years
- Marketing consent records: Duration of relationship plus 2 years
- Application/recruitment data: 1 year (unsuccessful applicants)
- Access logs and security data: 2 years
Data must be securely deleted or anonymized when the retention period expires.
Annual data retention audits are conducted by the DPO office.

SECTION 3: DATA SUBJECT RIGHTS
Under applicable regulations (GDPR, CCPA), individuals have the right to:
- Access their personal data (response within 30 days)
- Rectify inaccurate personal data
- Erase personal data ("right to be forgotten") where legally applicable
- Restrict or object to certain processing activities
- Data portability in machine-readable format
- Withdraw consent at any time
All data subject requests must be logged and processed through the Privacy Portal.
The DPO must be notified of all erasure and portability requests.

SECTION 4: DATA BREACH NOTIFICATION
In the event of a personal data breach:
- The DPO must be notified within 24 hours of discovery
- Regulatory authorities must be notified within 72 hours (if risk to individuals)
- Affected individuals must be notified without undue delay if high risk
- Breach documentation must include: nature, categories of data, approximate
  number of individuals affected, likely consequences, and remedial measures
- Post-breach review must be completed within 30 days
"""
    },
    {
        "doc_id": "HR-002",
        "title": "Remote Work and Flexible Arrangements Policy",
        "department": "Human Resources",
        "category": "remote_work",
        "effective_date": "2024-04-01",
        "version": "2.0",
        "text": """
SECTION 1: ELIGIBILITY
Remote work arrangements are available to employees who:
- Have completed their 90-day probationary period
- Maintain a performance rating of 3 (Meets Expectations) or above
- Work in a role that can be performed effectively from a remote location
- Have reliable internet (minimum 50 Mbps download / 10 Mbps upload)
- Have a dedicated, private workspace suitable for professional activities
Eligibility is reviewed quarterly. Remote work may be revoked if performance
declines below expectations or business needs require on-site presence.

SECTION 2: WORK SCHEDULE AND AVAILABILITY
Remote employees must:
- Maintain core hours of 10:00 AM to 3:00 PM in their local time zone
- Be available on Slack and email during core hours with response within 30 minutes
- Attend all scheduled video meetings with camera on
- Track work hours accurately in the time management system
- Take regular breaks and maintain work-life boundaries
Flexible scheduling outside core hours is permitted with manager agreement.
All remote employees must be on-site at least 2 days per month for team activities.

SECTION 3: EQUIPMENT AND WORKSPACE
The company provides the following for remote workers:
- Laptop computer (standard company issue)
- External monitor (up to 27 inches)
- Keyboard and mouse
- Headset for video calls
- $500 one-time home office setup stipend
Employees are responsible for their internet connection, desk, and chair.
An ergonomic assessment checklist must be completed and submitted to HR.
Company equipment must be returned within 5 business days of employment end.

SECTION 4: COMMUNICATION EXPECTATIONS
Remote employees must follow these communication standards:
- Daily standup updates posted in team Slack channel by 10:30 AM
- Weekly 1-on-1 video call with direct manager
- Monthly team all-hands meeting (in-person when possible)
- Use project management tools for task tracking and updates
- Document decisions and meeting notes in shared workspace
- Escalate blockers within 4 hours of identification
Asynchronous communication is preferred for non-urgent matters.
"""
    }
]

print(f"Created {len(POLICY_DOCUMENTS)} policy documents:")
for doc in POLICY_DOCUMENTS:
    text_len = len(doc["text"].strip())
    print(f"  [{doc['doc_id']}] {doc['title']}")
    print(f"       Department: {doc['department']}, Category: {doc['category']}, "
          f"Date: {doc['effective_date']}, Length: {text_len:,} chars")

Created 5 policy documents:
  [HR-001] Employee Handbook — Human Resources Policy
       Department: Human Resources, Category: employee_benefits, Date: 2024-01-15, Length: 2,069 chars
  [IT-001] Information Security Policy
       Department: IT Security, Category: access_control, Date: 2024-03-01, Length: 2,109 chars
  [FIN-001] Travel and Expense Policy
       Department: Finance, Category: expense_management, Date: 2024-06-01, Length: 1,888 chars
  [LEGAL-001] Data Privacy and Protection Policy
       Department: Legal, Category: data_privacy, Date: 2024-02-01, Length: 2,182 chars
  [HR-002] Remote Work and Flexible Arrangements Policy
       Department: Human Resources, Category: remote_work, Date: 2024-04-01, Length: 2,103 chars


## Document Loading with Metadata Enrichment

Each policy document is loaded with its metadata attached. This metadata will follow
each chunk through the pipeline, enabling:
- **Source tracking**: Which document does this chunk come from?
- **Department filtering**: Only search IT policies, or only HR policies
- **Temporal filtering**: Only policies effective after a certain date
- **Category filtering**: Only search "access_control" or "expense_management" chunks

In [4]:
# ── Load and structure documents with metadata ──

class PolicyDocument:
    """Represents a loaded policy document with metadata."""

    def __init__(self, doc_dict):
        self.doc_id = doc_dict["doc_id"]
        self.title = doc_dict["title"]
        self.department = doc_dict["department"]
        self.category = doc_dict["category"]
        self.effective_date = doc_dict["effective_date"]
        self.version = doc_dict["version"]
        self.text = doc_dict["text"].strip()
        self.sections = self._extract_sections()

    def _extract_sections(self):
        """Split document into named sections."""
        sections = []
        parts = re.split(r"(SECTION \d+:.*?)\n", self.text)
        current_title = "Preamble"
        current_text = ""
        for part in parts:
            if re.match(r"SECTION \d+:", part):
                if current_text.strip():
                    sections.append({"title": current_title, "text": current_text.strip()})
                current_title = part.strip()
                current_text = ""
            else:
                current_text += part
        if current_text.strip():
            sections.append({"title": current_title, "text": current_text.strip()})
        return sections

    @property
    def metadata(self):
        return {
            "doc_id": self.doc_id,
            "title": self.title,
            "department": self.department,
            "category": self.category,
            "effective_date": self.effective_date,
            "version": self.version,
        }

# Load all documents
documents = [PolicyDocument(d) for d in POLICY_DOCUMENTS]

print(f"Loaded {len(documents)} policy documents\n")
for doc in documents:
    print(f"  [{doc.doc_id}] {doc.title}")
    print(f"    Sections: {len(doc.sections)}")
    for sec in doc.sections:
        print(f"      - {sec['title']} ({len(sec['text'])} chars)")
    print()

Loaded 5 policy documents

  [HR-001] Employee Handbook — Human Resources Policy
    Sections: 4
      - SECTION 1: LEAVE POLICY (484 chars)
      - SECTION 2: CODE OF CONDUCT (462 chars)
      - SECTION 3: BENEFITS OVERVIEW (590 chars)
      - SECTION 4: PERFORMANCE REVIEWS (416 chars)

  [IT-001] Information Security Policy
    Sections: 4
      - SECTION 1: PASSWORD POLICY (489 chars)
      - SECTION 2: DATA CLASSIFICATION (545 chars)
      - SECTION 3: INCIDENT RESPONSE (552 chars)
      - SECTION 4: ACCEPTABLE USE (404 chars)

  [FIN-001] Travel and Expense Policy
    Sections: 4
      - SECTION 1: TRAVEL BOOKING (666 chars)
      - SECTION 2: MEAL ALLOWANCES (335 chars)
      - SECTION 3: EXPENSE REPORTING (465 chars)
      - SECTION 4: NON-TRAVEL EXPENSES (303 chars)

  [LEGAL-001] Data Privacy and Protection Policy
    Sections: 4
      - SECTION 1: DATA COLLECTION PRINCIPLES (546 chars)
      - SECTION 2: DATA RETENTION (524 chars)
      - SECTION 3: DATA SUBJECT RIGHTS (516 c

In [5]:
# ── Document corpus statistics ──

total_chars = sum(len(d.text) for d in documents)
total_sections = sum(len(d.sections) for d in documents)
departments = set(d.department for d in documents)
categories = set(d.category for d in documents)

print("=== Corpus Statistics ===")
print(f"  Documents: {len(documents)}")
print(f"  Total sections: {total_sections}")
print(f"  Total characters: {total_chars:,}")
print(f"  Departments: {departments}")
print(f"  Categories: {categories}")
print(f"  Date range: {min(d.effective_date for d in documents)} to "
      f"{max(d.effective_date for d in documents)}")

=== Corpus Statistics ===
  Documents: 5
  Total sections: 20
  Total characters: 10,351
  Departments: {'Finance', 'Legal', 'Human Resources', 'IT Security'}
  Categories: {'expense_management', 'remote_work', 'data_privacy', 'employee_benefits', 'access_control'}
  Date range: 2024-01-15 to 2024-06-01


## Chunking Strategy with Metadata Preservation

Chunking is critical for RAG quality. Our strategy:
1. **Section-aware chunking** — split on section boundaries first, then by size
2. **Overlap** — chunks overlap to preserve context at boundaries
3. **Metadata inheritance** — every chunk inherits its parent document's metadata
   plus section-level metadata (section title, chunk position)
4. **Chunk IDs** — deterministic IDs for citation tracking

### Why section-aware chunking?
Naive character-level chunking can split a sentence about password requirements
across two chunks, losing meaning. Section-aware chunking respects document
structure, keeping related content together.

In [6]:
# ── Section-aware chunking with metadata preservation ──

class Chunk:
    """A text chunk with full metadata lineage."""

    def __init__(self, text, doc_metadata, section_title, chunk_index, total_chunks):
        self.text = text
        self.doc_id = doc_metadata["doc_id"]
        self.doc_title = doc_metadata["title"]
        self.department = doc_metadata["department"]
        self.category = doc_metadata["category"]
        self.effective_date = doc_metadata["effective_date"]
        self.version = doc_metadata["version"]
        self.section_title = section_title
        self.chunk_index = chunk_index
        self.total_chunks = total_chunks
        # Deterministic chunk ID
        content_hash = hashlib.md5(f"{self.doc_id}:{section_title}:{chunk_index}".encode()).hexdigest()[:8]
        self.chunk_id = f"{self.doc_id}-{content_hash}"

    @property
    def metadata(self):
        return {
            "chunk_id": self.chunk_id,
            "doc_id": self.doc_id,
            "doc_title": self.doc_title,
            "department": self.department,
            "category": self.category,
            "effective_date": self.effective_date,
            "version": self.version,
            "section_title": self.section_title,
            "chunk_index": self.chunk_index,
            "total_chunks": self.total_chunks,
        }

    @property
    def citation(self):
        return f"[{self.doc_id}] {self.doc_title}, {self.section_title} (chunk {self.chunk_index+1}/{self.total_chunks})"


def chunk_document(doc, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Chunk a PolicyDocument into Chunk objects with metadata."""
    chunks = []
    for section in doc.sections:
        text = section["text"]
        section_title = section["title"]

        if len(text) <= chunk_size:
            chunks.append(Chunk(text, doc.metadata, section_title, 0, 1))
        else:
            # Sliding window with overlap
            start = 0
            chunk_list = []
            while start < len(text):
                end = start + chunk_size
                chunk_text = text[start:end]
                chunk_list.append(chunk_text)
                start += chunk_size - overlap

            for i, ct in enumerate(chunk_list):
                chunks.append(Chunk(ct.strip(), doc.metadata, section_title, i, len(chunk_list)))

    return chunks


# Chunk all documents
all_chunks = []
for doc in documents:
    doc_chunks = chunk_document(doc)
    all_chunks.extend(doc_chunks)
    print(f"[{doc.doc_id}] {doc.title}: {len(doc_chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")
print(f"Avg chunk length: {np.mean([len(c.text) for c in all_chunks]):.0f} chars")
print(f"Min/Max chunk length: {min(len(c.text) for c in all_chunks)} / "
      f"{max(len(c.text) for c in all_chunks)} chars")

[HR-001] Employee Handbook — Human Resources Policy: 5 chunks
[IT-001] Information Security Policy: 6 chunks
[FIN-001] Travel and Expense Policy: 5 chunks
[LEGAL-001] Data Privacy and Protection Policy: 7 chunks
[HR-002] Remote Work and Flexible Arrangements Policy: 6 chunks

Total chunks: 29
Avg chunk length: 366 chars
Min/Max chunk length: 105 / 500 chars


In [7]:
# ── Inspect sample chunks to verify metadata preservation ──

print("=== Sample Chunks (first 3) ===\n")
for chunk in all_chunks[:3]:
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Citation: {chunk.citation}")
    print(f"Department: {chunk.department} | Category: {chunk.category}")
    print(f"Effective date: {chunk.effective_date} | Version: {chunk.version}")
    print(f"Text preview: {chunk.text[:150]}...")
    print("-" * 70)

=== Sample Chunks (first 3) ===

Chunk ID: HR-001-ad65f921
Citation: [HR-001] Employee Handbook — Human Resources Policy, SECTION 1: LEAVE POLICY (chunk 1/1)
Department: Human Resources | Category: employee_benefits
Effective date: 2024-01-15 | Version: 3.2
Text preview: Full-time employees are entitled to paid time off (PTO) as follows:
- 0-2 years of service: 15 days per year
- 3-5 years of service: 20 days per year
...
----------------------------------------------------------------------
Chunk ID: HR-001-963c51f2
Citation: [HR-001] Employee Handbook — Human Resources Policy, SECTION 2: CODE OF CONDUCT (chunk 1/1)
Department: Human Resources | Category: employee_benefits
Effective date: 2024-01-15 | Version: 3.2
Text preview: All employees must maintain professional behavior at all times. This includes:
- Respectful communication with colleagues and clients
- Appropriate bu...
----------------------------------------------------------------------
Chunk ID: HR-001-03d9b0ad
Citation: 

## Embedding and Vector Store

We now embed all chunks into dense vectors for semantic search.

**Primary path:** sentence-transformers (`all-MiniLM-L6-v2`) + ChromaDB with metadata filtering.

**Fallback path:** TF-IDF vectorization + cosine similarity with manual metadata filtering.

### Why ChromaDB?
ChromaDB supports **metadata filtering natively** — you can pass `where` clauses
to filter chunks by department, category, or date before computing similarity.
This is more efficient than retrieving all chunks and filtering post-hoc.

In [8]:
# ── Build embedding index ──

class VectorStore:
    """Unified vector store with metadata filtering."""

    def __init__(self, chunks):
        self.chunks = chunks
        self.texts = [c.text for c in chunks]
        self._index_built = False

        if USE_SENTENCE_TRANSFORMERS and USE_CHROMA:
            self._build_chroma_index()
        else:
            self._build_tfidf_index()

    def _build_chroma_index(self):
        """Build ChromaDB index with sentence-transformer embeddings."""
        print("Building ChromaDB index with sentence-transformers...")
        self.model = SentenceTransformer(EMBEDDING_MODEL)
        self.client = chromadb.Client()

        # Delete collection if it exists (idempotent)
        try:
            self.client.delete_collection("policies")
        except Exception:
            pass

        self.collection = self.client.create_collection(
            name="policies",
            metadata={"hnsw:space": "cosine"}
        )

        # Embed and add to collection
        embeddings = self.model.encode(self.texts, show_progress_bar=False).tolist()
        ids = [c.chunk_id for c in self.chunks]
        metadatas = [c.metadata for c in self.chunks]

        self.collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=self.texts,
            metadatas=metadatas,
        )
        self.backend = "chroma"
        self._index_built = True
        print(f"ChromaDB index built: {self.collection.count()} chunks indexed")

    def _build_tfidf_index(self):
        """Build TF-IDF fallback index."""
        print("Building TF-IDF fallback index...")
        self.vectorizer = TfidfVectorizer(
            max_features=5000,
            stop_words="english",
            ngram_range=(1, 2),
        )
        self.tfidf_matrix = self.vectorizer.fit_transform(self.texts)
        self.backend = "tfidf"
        self._index_built = True
        print(f"TF-IDF index built: {self.tfidf_matrix.shape}")

    def search(self, query, top_k=TOP_K, metadata_filter=None):
        """Search for relevant chunks with optional metadata filtering.

        Args:
            query: Search query string
            top_k: Number of results to return
            metadata_filter: Dict of metadata key-value pairs to filter on
                e.g., {"department": "IT Security"} or {"category": "access_control"}

        Returns:
            List of (chunk, score) tuples sorted by relevance
        """
        if self.backend == "chroma":
            return self._search_chroma(query, top_k, metadata_filter)
        else:
            return self._search_tfidf(query, top_k, metadata_filter)

    def _search_chroma(self, query, top_k, metadata_filter):
        """ChromaDB search with native metadata filtering."""
        where_clause = None
        if metadata_filter:
            if len(metadata_filter) == 1:
                key, val = list(metadata_filter.items())[0]
                where_clause = {key: val}
            else:
                where_clause = {
                    "$and": [{k: v} for k, v in metadata_filter.items()]
                }

        query_embedding = self.model.encode([query]).tolist()
        results = self.collection.query(
            query_embeddings=query_embedding,
            n_results=min(top_k, self.collection.count()),
            where=where_clause,
        )

        # Map results back to Chunk objects
        chunk_map = {c.chunk_id: c for c in self.chunks}
        output = []
        for i, chunk_id in enumerate(results["ids"][0]):
            chunk = chunk_map[chunk_id]
            # ChromaDB returns distances; convert to similarity
            distance = results["distances"][0][i]
            similarity = 1.0 - distance  # cosine distance to similarity
            output.append((chunk, similarity))

        return output

    def _search_tfidf(self, query, top_k, metadata_filter):
        """TF-IDF search with manual metadata filtering."""
        # Apply metadata filter first
        if metadata_filter:
            candidate_indices = []
            for i, chunk in enumerate(self.chunks):
                match = all(
                    getattr(chunk, k, chunk.metadata.get(k)) == v
                    for k, v in metadata_filter.items()
                )
                if match:
                    candidate_indices.append(i)
        else:
            candidate_indices = list(range(len(self.chunks)))

        if not candidate_indices:
            return []

        # Compute similarities
        query_vec = self.vectorizer.transform([query])
        candidate_matrix = self.tfidf_matrix[candidate_indices]
        similarities = cosine_similarity(query_vec, candidate_matrix).flatten()

        # Rank and return top-k
        top_indices = np.argsort(similarities)[::-1][:top_k]
        output = []
        for idx in top_indices:
            real_idx = candidate_indices[idx]
            output.append((self.chunks[real_idx], float(similarities[idx])))

        return output


# Build the vector store
vector_store = VectorStore(all_chunks)

Building ChromaDB index with sentence-transformers...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ChromaDB index built: 29 chunks indexed


## Baseline: Keyword Search

Before evaluating our RAG pipeline, we establish a baseline using simple
keyword matching. This helps us measure how much semantic search improves
retrieval quality over naive pattern matching.

In [9]:
# ── Baseline: keyword search ──

def keyword_search(query, chunks, top_k=TOP_K):
    """Simple keyword matching baseline — counts query term occurrences."""
    query_terms = set(query.lower().split())
    scored = []
    for chunk in chunks:
        text_lower = chunk.text.lower()
        # Count how many query terms appear in the chunk
        matches = sum(1 for term in query_terms if term in text_lower)
        score = matches / max(len(query_terms), 1)
        scored.append((chunk, score))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]


# Test baseline
test_query = "How many vacation days do I get after 3 years?"
baseline_results = keyword_search(test_query, all_chunks)

print(f"Query: {test_query}")
print(f"\n--- Baseline (Keyword Search) Top {TOP_K} ---")
for chunk, score in baseline_results:
    print(f"  Score: {score:.2f} | {chunk.citation}")
    print(f"  Preview: {chunk.text[:120]}...")
    print()

Query: How many vacation days do I get after 3 years?

--- Baseline (Keyword Search) Top 5 ---
  Score: 0.50 | [FIN-001] Travel and Expense Policy, SECTION 3: EXPENSE REPORTING (chunk 1/1)
  Preview: Expense reports must be submitted within 30 days of travel completion.
Reports submitted after 60 days will not be reimb...

  Score: 0.40 | [FIN-001] Travel and Expense Policy, SECTION 1: TRAVEL BOOKING (chunk 1/2)
  Preview: All business travel must be pre-approved by the employee's direct manager.
Travel exceeding $2,000 requires VP-level app...

  Score: 0.40 | [LEGAL-001] Data Privacy and Protection Policy, SECTION 4: DATA BREACH NOTIFICATION (chunk 1/1)
  Preview: In the event of a personal data breach:
- The DPO must be notified within 24 hours of discovery
- Regulatory authorities...

  Score: 0.30 | [HR-001] Employee Handbook — Human Resources Policy, SECTION 1: LEAVE POLICY (chunk 1/1)
  Preview: Full-time employees are entitled to paid time off (PTO) as follows:
- 0-2 years of s

## Semantic Retrieval with Metadata Filtering

Now we test the vector-based retrieval, which understands meaning rather than
just matching keywords. We can also apply metadata filters to narrow the search
to specific departments or categories.

In [10]:
# ── Semantic retrieval (no filter) ──

print(f"Query: {test_query}")
print(f"\n--- Semantic Retrieval (No Filter) Top {TOP_K} ---")
semantic_results = vector_store.search(test_query, top_k=TOP_K)
for chunk, score in semantic_results:
    print(f"  Score: {score:.3f} | {chunk.citation}")
    print(f"  Dept: {chunk.department} | Category: {chunk.category}")
    print(f"  Preview: {chunk.text[:120]}...")
    print()

Query: How many vacation days do I get after 3 years?

--- Semantic Retrieval (No Filter) Top 5 ---
  Score: 0.332 | [HR-001] Employee Handbook — Human Resources Policy, SECTION 1: LEAVE POLICY (chunk 1/1)
  Dept: Human Resources | Category: employee_benefits
  Preview: Full-time employees are entitled to paid time off (PTO) as follows:
- 0-2 years of service: 15 days per year
- 3-5 years...

  Score: 0.313 | [HR-001] Employee Handbook — Human Resources Policy, SECTION 3: BENEFITS OVERVIEW (chunk 2/2)
  Dept: Human Resources | Category: employee_benefits
  Preview: for primary caregivers, 4 weeks for secondary
Open enrollment occurs annually in November. Changes outside open enrollme...

  Score: 0.244 | [HR-002] Remote Work and Flexible Arrangements Policy, SECTION 2: WORK SCHEDULE AND AVAILABILITY (chunk 2/2)
  Dept: Human Resources | Category: remote_work
  Preview: th manager agreement.
All remote employees must be on-site at least 2 days per month for team activities....

  Score:

In [11]:
# ── Semantic retrieval WITH metadata filter ──

print(f"Query: {test_query}")
print(f"Filter: department = 'Human Resources'")
print(f"\n--- Filtered Semantic Retrieval Top {TOP_K} ---")
filtered_results = vector_store.search(
    test_query,
    top_k=TOP_K,
    metadata_filter={"department": "Human Resources"}
)
for chunk, score in filtered_results:
    print(f"  Score: {score:.3f} | {chunk.citation}")
    print(f"  Dept: {chunk.department} | Category: {chunk.category}")
    print(f"  Preview: {chunk.text[:120]}...")
    print()

Query: How many vacation days do I get after 3 years?
Filter: department = 'Human Resources'

--- Filtered Semantic Retrieval Top 5 ---
  Score: 0.332 | [HR-001] Employee Handbook — Human Resources Policy, SECTION 1: LEAVE POLICY (chunk 1/1)
  Dept: Human Resources | Category: employee_benefits
  Preview: Full-time employees are entitled to paid time off (PTO) as follows:
- 0-2 years of service: 15 days per year
- 3-5 years...

  Score: 0.313 | [HR-001] Employee Handbook — Human Resources Policy, SECTION 3: BENEFITS OVERVIEW (chunk 2/2)
  Dept: Human Resources | Category: employee_benefits
  Preview: for primary caregivers, 4 weeks for secondary
Open enrollment occurs annually in November. Changes outside open enrollme...

  Score: 0.244 | [HR-002] Remote Work and Flexible Arrangements Policy, SECTION 2: WORK SCHEDULE AND AVAILABILITY (chunk 2/2)
  Dept: Human Resources | Category: remote_work
  Preview: th manager agreement.
All remote employees must be on-site at least 2 days per mon

## Citation Extraction and Source Chunk Display

A key requirement for trustworthy RAG is **citation**. The user needs to see:
1. **Which document(s)** the answer came from
2. **Which section** within the document
3. **The exact text chunk** used to generate the answer
4. **A confidence indicator** about how well the chunks match the question

This transparency lets users verify answers against the source material.

In [12]:
# ── Citation and source display formatting ──

def format_citations(results, max_chunks=3):
    """Format retrieved chunks with full citation information."""
    lines = []
    lines.append("=" * 70)
    lines.append("SOURCES AND CITATIONS")
    lines.append("=" * 70)

    seen_docs = set()
    for i, (chunk, score) in enumerate(results[:max_chunks]):
        lines.append(f"\n--- Source {i+1} (relevance: {score:.3f}) ---")
        lines.append(f"  Document:  {chunk.doc_title}")
        lines.append(f"  Doc ID:    {chunk.doc_id}")
        lines.append(f"  Section:   {chunk.section_title}")
        lines.append(f"  Dept:      {chunk.department}")
        lines.append(f"  Category:  {chunk.category}")
        lines.append(f"  Effective: {chunk.effective_date} (v{chunk.version})")
        lines.append(f"  Chunk:     {chunk.chunk_index+1} of {chunk.total_chunks}")
        lines.append(f"  Chunk ID:  {chunk.chunk_id}")
        lines.append(f"\n  Full text:")
        # Wrap text for display
        wrapped = textwrap.fill(chunk.text, width=70, initial_indent="    ", subsequent_indent="    ")
        lines.append(wrapped)
        seen_docs.add(chunk.doc_id)

    lines.append(f"\n{'=' * 70}")
    lines.append(f"Documents referenced: {', '.join(sorted(seen_docs))}")
    lines.append(f"Total sources shown: {min(max_chunks, len(results))}")
    return "\n".join(lines)


# Demo citation display
results = vector_store.search(test_query, top_k=3)
print(format_citations(results))

SOURCES AND CITATIONS

--- Source 1 (relevance: 0.332) ---
  Document:  Employee Handbook — Human Resources Policy
  Doc ID:    HR-001
  Section:   SECTION 1: LEAVE POLICY
  Dept:      Human Resources
  Category:  employee_benefits
  Effective: 2024-01-15 (v3.2)
  Chunk:     1 of 1
  Chunk ID:  HR-001-ad65f921

  Full text:
    Full-time employees are entitled to paid time off (PTO) as
    follows: - 0-2 years of service: 15 days per year - 3-5 years of
    service: 20 days per year - 6-10 years of service: 25 days per
    year - 10+ years of service: 30 days per year PTO requests must be
    submitted at least 2 weeks in advance through the HR portal.
    Unused PTO up to 5 days may be carried over to the next calendar
    year. Any PTO beyond 5 days will be forfeited on December 31st
    unless approved by department management.

--- Source 2 (relevance: 0.313) ---
  Document:  Employee Handbook — Human Resources Policy
  Doc ID:    HR-001
  Section:   SECTION 3: BENEFITS OVERVIEW
  

## Confidence Scoring

Not all retrieval results are equally reliable. Our confidence scorer evaluates:

| Factor | What it measures | Weight |
|--------|-----------------|--------|
| **Top score** | How similar is the best chunk to the query? | 40% |
| **Score gap** | How much better is the top chunk vs. the rest? | 20% |
| **Chunk agreement** | Do multiple chunks agree on the topic? | 20% |
| **Source diversity** | Do multiple documents support the answer? | 20% |

**Interpretation:**
- **HIGH** (> 0.7): Strong evidence from multiple relevant chunks
- **MEDIUM** (0.5–0.7): Some evidence, but may need verification
- **LOW** (< 0.5): Weak evidence — answer may be unreliable

In [13]:
# ── Confidence scoring ──

def compute_confidence(results, threshold=CONFIDENCE_THRESHOLD):
    """Compute retrieval confidence score from search results."""
    if not results:
        return {"score": 0.0, "level": "NONE", "factors": {}}

    scores = [s for _, s in results]
    top_score = scores[0]
    avg_score = np.mean(scores)

    # Factor 1: Top chunk relevance (40%)
    top_factor = min(top_score / 0.8, 1.0)  # normalize against good threshold

    # Factor 2: Score gap — is there a clear winner? (20%)
    if len(scores) > 1:
        gap = scores[0] - scores[1]
        gap_factor = min(gap / 0.2, 1.0)  # normalize
    else:
        gap_factor = 1.0

    # Factor 3: Chunk agreement — do top chunks agree on topic? (20%)
    doc_ids = [c.doc_id for c, _ in results[:3]]
    sections = [c.section_title for c, _ in results[:3]]
    # Higher agreement if chunks come from similar sections
    unique_sections = len(set(sections))
    agreement_factor = 1.0 / max(unique_sections, 1)

    # Factor 4: Multi-document support (20%)
    unique_docs = len(set(doc_ids))
    diversity_factor = min(unique_docs / 2.0, 1.0)  # 2+ docs = max score

    # Weighted combination
    final_score = (
        0.40 * top_factor +
        0.20 * gap_factor +
        0.20 * agreement_factor +
        0.20 * diversity_factor
    )

    if final_score >= 0.7:
        level = "HIGH"
    elif final_score >= threshold:
        level = "MEDIUM"
    else:
        level = "LOW"

    return {
        "score": round(final_score, 3),
        "level": level,
        "factors": {
            "top_relevance": round(top_factor, 3),
            "score_gap": round(gap_factor, 3),
            "chunk_agreement": round(agreement_factor, 3),
            "source_diversity": round(diversity_factor, 3),
        }
    }


# Test confidence scoring
confidence = compute_confidence(results)
print(f"Query: {test_query}")
print(f"\nConfidence: {confidence['score']:.3f} ({confidence['level']})")
print(f"  Top relevance:    {confidence['factors']['top_relevance']:.3f}")
print(f"  Score gap:        {confidence['factors']['score_gap']:.3f}")
print(f"  Chunk agreement:  {confidence['factors']['chunk_agreement']:.3f}")
print(f"  Source diversity:  {confidence['factors']['source_diversity']:.3f}")

Query: How many vacation days do I get after 3 years?

Confidence: 0.452 (LOW)
  Top relevance:    0.415
  Score gap:        0.096
  Chunk agreement:  0.333
  Source diversity:  1.000


## Full RAG Pipeline

Now we combine everything into a single `PolicyAssistant` class that handles:
1. Query processing
2. Optional metadata filtering
3. Semantic retrieval
4. Citation extraction
5. Confidence scoring
6. Answer synthesis (rule-based summarization of retrieved chunks)

In [14]:
# ── Full Policy Assistant Pipeline ──

class PolicyAssistant:
    """Multi-document RAG policy assistant with citations and confidence."""

    def __init__(self, vector_store):
        self.vector_store = vector_store

    def answer(self, query, top_k=TOP_K, metadata_filter=None, show_sources=True):
        """Answer a policy question with citations and confidence.

        Args:
            query: Natural language question
            top_k: Number of chunks to retrieve
            metadata_filter: Optional dict to filter by metadata
            show_sources: Whether to display source chunks

        Returns:
            Dict with answer, citations, confidence
        """
        # Step 1: Retrieve relevant chunks
        results = self.vector_store.search(query, top_k=top_k, metadata_filter=metadata_filter)

        if not results:
            return {
                "answer": "No relevant policy information found for this query.",
                "citations": [],
                "confidence": {"score": 0.0, "level": "NONE"},
            }

        # Step 2: Synthesize answer from top chunks
        answer_text = self._synthesize(query, results)

        # Step 3: Extract citations
        citations = [
            {
                "source": chunk.citation,
                "doc_id": chunk.doc_id,
                "section": chunk.section_title,
                "relevance": round(score, 3),
            }
            for chunk, score in results[:3]
        ]

        # Step 4: Compute confidence
        confidence = compute_confidence(results)

        # Step 5: Display results
        if show_sources:
            self._display_results(query, answer_text, citations, confidence, results, metadata_filter)

        return {
            "answer": answer_text,
            "citations": citations,
            "confidence": confidence,
        }

    def _synthesize(self, query, results):
        """Extract and combine relevant information from retrieved chunks."""
        # Use the top chunks to build an answer
        top_chunks = results[:3]
        answer_parts = []

        for chunk, score in top_chunks:
            # Extract the most relevant sentences from each chunk
            sentences = re.split(r'(?<=[.!?])\s+', chunk.text)
            query_terms = set(query.lower().split())

            # Score each sentence by query term overlap
            scored_sentences = []
            for sent in sentences:
                sent_lower = sent.lower()
                overlap = sum(1 for t in query_terms if t in sent_lower)
                if overlap > 0:
                    scored_sentences.append((sent, overlap))

            scored_sentences.sort(key=lambda x: x[1], reverse=True)
            best = [s for s, _ in scored_sentences[:3]]

            if best:
                source_note = f"[{chunk.doc_id}, {chunk.section_title}]"
                answer_parts.append(f"{' '.join(best)} {source_note}")

        if answer_parts:
            return "\n\n".join(answer_parts)
        else:
            # Fallback: return top chunk text
            return f"{results[0][0].text[:300]}... [{results[0][0].doc_id}]"

    def _display_results(self, query, answer, citations, confidence, results, metadata_filter):
        """Pretty-print the full response."""
        print("=" * 70)
        print(f"QUERY: {query}")
        if metadata_filter:
            filters = ", ".join(f"{k}={v}" for k, v in metadata_filter.items())
            print(f"FILTER: {filters}")
        print("=" * 70)

        print(f"\nANSWER:")
        print(textwrap.fill(answer, width=70, initial_indent="  ", subsequent_indent="  "))

        print(f"\nCONFIDENCE: {confidence['score']:.3f} ({confidence['level']})")
        for factor, value in confidence["factors"].items():
            bar = "#" * int(value * 20)
            print(f"  {factor:20s}: {value:.3f} |{bar}|")

        print(f"\nCITATIONS:")
        for i, cit in enumerate(citations, 1):
            print(f"  [{i}] {cit['source']} (relevance: {cit['relevance']:.3f})")

        print(f"\nSOURCE CHUNKS:")
        for i, (chunk, score) in enumerate(results[:3], 1):
            print(f"  --- Chunk {i} (score: {score:.3f}) ---")
            preview = textwrap.fill(chunk.text[:200], width=66, initial_indent="    ", subsequent_indent="    ")
            print(preview)
            if len(chunk.text) > 200:
                print("    ...")

        print("=" * 70)


# Create the assistant
assistant = PolicyAssistant(vector_store)
print("Policy Assistant ready.")

Policy Assistant ready.


## Query Examples

Let's test the full pipeline with various questions spanning different policy areas.

In [15]:
# ── Query 1: HR Leave Policy ──
assistant.answer("How many vacation days do I get after 3 years of service?")

QUERY: How many vacation days do I get after 3 years of service?

ANSWER:
  Full-time employees are entitled to paid time off (PTO) as follows:
  - 0-2 years of service: 15 days per year - 3-5 years of service: 20
  days per year - 6-10 years of service: 25 days per year - 10+ years
  of service: 30 days per year PTO requests must be submitted at least
  2 weeks in advance through the HR portal. Any PTO beyond 5 days will
  be forfeited on December 31st unless approved by department
  management. Unused PTO up to 5 days may be carried over to the next
  calendar year. [HR-001, SECTION 1: LEAVE POLICY]  Changes outside
  open enrollment require a qualifying life event (marriage, birth,
  adoption, etc.). for primary caregivers, 4 weeks for secondary Open
  enrollment occurs annually in November. [HR-001, SECTION 3: BENEFITS
  OVERVIEW]  All remote employees must be on-site at least 2 days per
  month for team activities. [HR-002, SECTION 2: WORK SCHEDULE AND
  AVAILABILITY]

CONFIDENCE:

{'answer': 'Full-time employees are entitled to paid time off (PTO) as follows:\n- 0-2 years of service: 15 days per year\n- 3-5 years of service: 20 days per year\n- 6-10 years of service: 25 days per year\n- 10+ years of service: 30 days per year\nPTO requests must be submitted at least 2 weeks in advance through the HR portal. Any PTO beyond\n5 days will be forfeited on December 31st unless approved by department management. Unused PTO up to 5 days may be carried over to the next calendar year. [HR-001, SECTION 1: LEAVE POLICY]\n\nChanges outside open enrollment\nrequire a qualifying life event (marriage, birth, adoption, etc.). for primary caregivers, 4 weeks for secondary\nOpen enrollment occurs annually in November. [HR-001, SECTION 3: BENEFITS OVERVIEW]\n\nAll remote employees must be on-site at least 2 days per month for team activities. [HR-002, SECTION 2: WORK SCHEDULE AND AVAILABILITY]',
 'citations': [{'source': '[HR-001] Employee Handbook — Human Resources Policy, SECTION 

In [16]:
# ── Query 2: IT Security ──
assistant.answer("What is the password policy and how often must passwords be changed?")

QUERY: What is the password policy and how often must passwords be changed?

ANSWER:
  All system passwords must meet these requirements: - Minimum 14
  characters in length - Must contain uppercase, lowercase, numbers,
  and special characters - Passwords must be changed every 90 days -
  Previous 12 passwords cannot be reused - Multi-factor authentication
  (MFA) is mandatory for all systems - Passwords must not contain
  personal information or dictionary words Account lockout occurs
  after 5 failed login attempts. [IT-001, SECTION 1: PASSWORD POLICY]
  Under applicable regulations (GDPR, CCPA), individuals have the
  right to: - Access their personal data (response within 30 days) -
  Rectify inaccurate personal data - Erase personal data ("right to be
  forgotten") where legally applicable - Restrict or object to certain
  processing activities - Data portability in machine-readable format
  - Withdraw consent at any time All data subject requests must be
  logged and processed t

{'answer': 'All system passwords must meet these requirements:\n- Minimum 14 characters in length\n- Must contain uppercase, lowercase, numbers, and special characters\n- Passwords must be changed every 90 days\n- Previous 12 passwords cannot be reused\n- Multi-factor authentication (MFA) is mandatory for all systems\n- Passwords must not contain personal information or dictionary words\nAccount lockout occurs after 5 failed login attempts. [IT-001, SECTION 1: PASSWORD POLICY]\n\nUnder applicable regulations (GDPR, CCPA), individuals have the right to:\n- Access their personal data (response within 30 days)\n- Rectify inaccurate personal data\n- Erase personal data ("right to be forgotten") where legally applicable\n- Restrict or object to certain processing activities\n- Data portability in machine-readable format\n- Withdraw consent at any time\nAll data subject requests must be logged and processed through the Privacy Portal. The DPO must be notified of all erasure and porta [LEGAL-

In [17]:
# ── Query 3: Travel Expenses ──
assistant.answer("Can I expense meals during business travel? What are the limits?")

QUERY: Can I expense meals during business travel? What are the limits?

ANSWER:
  Meal expenses during business travel are reimbursed up to these
  daily limits: - Breakfast: $20 - Lunch: $30 - Dinner: $50 - Total
  daily maximum: $100 Alcohol is not reimbursable. Tips up to 20% are
  reimbursable. Receipts required for all meals over $25. [FIN-001,
  SECTION 2: MEAL ALLOWANCES]  All business travel must be pre-
  approved by the employee's direct manager. Bookings should be made
  through the corporate travel portal (TravelWorks) at least 14 days
  in advance. - Air travel: Economy class for flights under 6 hours;
  business class permitted   for flights over 6 hours with VP approval
  - Hotels: Maximum $250 per night in standard markets; $350 per night
  in high-cost   cities (NYC, SF, London, Tokyo). [FIN-001, SECTION 1:
  TRAVEL BOOKING]  Non-travel business expenses require pre-approval:
  - Office supplies under $100: No approval needed -
  Software/subscriptions under $500: Man

{'answer': "Meal expenses during business travel are reimbursed up to these daily limits:\n- Breakfast: $20\n- Lunch: $30\n- Dinner: $50\n- Total daily maximum: $100\nAlcohol is not reimbursable. Tips up to 20% are reimbursable. Receipts required for all meals over $25. [FIN-001, SECTION 2: MEAL ALLOWANCES]\n\nAll business travel must be pre-approved by the employee's direct manager. Bookings should be made\nthrough the corporate travel portal (TravelWorks) at least 14 days in advance. - Air travel: Economy class for flights under 6 hours; business class permitted\n  for flights over 6 hours with VP approval\n- Hotels: Maximum $250 per night in standard markets; $350 per night in high-cost\n  cities (NYC, SF, London, Tokyo). [FIN-001, SECTION 1: TRAVEL BOOKING]\n\nNon-travel business expenses require pre-approval:\n- Office supplies under $100: No approval needed\n- Software/subscriptions under $500: Manager approval\n- Equipment/hardware under $2,000: Director approval\n- Any expense 

In [18]:
# ── Query 4: Data Privacy ──
assistant.answer("How long does the company retain employee personal data?")

QUERY: How long does the company retain employee personal data?

ANSWER:
  Personal data is retained only as long as necessary for its original
  purpose or as required by law. Standard retention periods: -
  Employee records: 7 years after termination - Customer transaction
  data: 5 years - Marketing consent records: Duration of relationship
  plus 2 years - Application/recruitment data: 1 year (unsuccessful
  applicants) - Access logs and security data: 2 years Data must be
  securely deleted or anonymized when the retention period expires.
  [LEGAL-001, SECTION 2: DATA RETENTION]  urely deleted or anonymized
  when the retention period expires. Annual data retention audits are
  conducted by the DPO office. [LEGAL-001, SECTION 2: DATA RETENTION]
  Under applicable regulations (GDPR, CCPA), individuals have the
  right to: - Access their personal data (response within 30 days) -
  Rectify inaccurate personal data - Erase personal data ("right to be
  forgotten") where legally applic

{'answer': 'Personal data is retained only as long as necessary for its original purpose\nor as required by law. Standard retention periods:\n- Employee records: 7 years after termination\n- Customer transaction data: 5 years\n- Marketing consent records: Duration of relationship plus 2 years\n- Application/recruitment data: 1 year (unsuccessful applicants)\n- Access logs and security data: 2 years\nData must be securely deleted or anonymized when the retention period expires. [LEGAL-001, SECTION 2: DATA RETENTION]\n\nurely deleted or anonymized when the retention period expires. Annual data retention audits are conducted by the DPO office. [LEGAL-001, SECTION 2: DATA RETENTION]\n\nUnder applicable regulations (GDPR, CCPA), individuals have the right to:\n- Access their personal data (response within 30 days)\n- Rectify inaccurate personal data\n- Erase personal data ("right to be forgotten") where legally applicable\n- Restrict or object to certain processing activities\n- Data portab

In [19]:
# ── Query 5: Remote Work ──
assistant.answer("Am I eligible for remote work and what equipment is provided?")

QUERY: Am I eligible for remote work and what equipment is provided?

ANSWER:
  Remote work arrangements are available to employees who: - Have
  completed their 90-day probationary period - Maintain a performance
  rating of 3 (Meets Expectations) or above - Work in a role that can
  be performed effectively from a remote location - Have reliable
  internet (minimum 50 Mbps download / 10 Mbps upload) - Have a
  dedicated, private workspace suitable for professional activities
  Eligibility is reviewed quarterly. Remote work may be revoked if
  performance declines below expectations or [HR-002, SECTION 1:
  ELIGIBILITY]  The company provides the following for remote workers:
  - Laptop computer (standard company issue) - External monitor (up to
  27 inches) - Keyboard and mouse - Headset for video calls - $500
  one-time home office setup stipend Employees are responsible for
  their internet connection, desk, and chair. An ergonomic assessment
  checklist must be completed and submit

{'answer': 'Remote work arrangements are available to employees who:\n- Have completed their 90-day probationary period\n- Maintain a performance rating of 3 (Meets Expectations) or above\n- Work in a role that can be performed effectively from a remote location\n- Have reliable internet (minimum 50 Mbps download / 10 Mbps upload)\n- Have a dedicated, private workspace suitable for professional activities\nEligibility is reviewed quarterly. Remote work may be revoked if performance\ndeclines below expectations or [HR-002, SECTION 1: ELIGIBILITY]\n\nThe company provides the following for remote workers:\n- Laptop computer (standard company issue)\n- External monitor (up to 27 inches)\n- Keyboard and mouse\n- Headset for video calls\n- $500 one-time home office setup stipend\nEmployees are responsible for their internet connection, desk, and chair. An ergonomic assessment checklist must be completed and submitted to HR. Company equipment must be returned within 5 business days of employm

## Metadata Filtering in Action

One of the most powerful capabilities of multi-document RAG is **metadata filtering**.
Instead of searching across all documents, you can target specific departments,
categories, or time periods. This dramatically improves precision for scoped queries.

In [20]:
# ── Filter by department ──
print("=== Query scoped to IT Security only ===\n")
assistant.answer(
    "What happens if I share my credentials?",
    metadata_filter={"department": "IT Security"}
)

=== Query scoped to IT Security only ===

QUERY: What happens if I share my credentials?
FILTER: department=IT Security

ANSWER:
  Locked accounts require IT Service Desk verification to unlock. All
  system passwords must meet these requirements: - Minimum 14
  characters in length - Must contain uppercase, lowercase, numbers,
  and special characters - Passwords must be changed every 90 days -
  Previous 12 passwords cannot be reused - Multi-factor authentication
  (MFA) is mandatory for all systems - Passwords must not contain
  personal information or dictionary words Account lockout occurs
  after 5 failed login attempts. [IT-001, SECTION 1: PASSWORD POLICY]
  Limited personal use is permitted if it does not interfere with work
  duties or violate any policy. Company IT resources are provided for
  business purposes. Prohibited activities include: unauthorized
  software installation, cryptocurrency mining, accessing
  inappropriate content, using company email for personal busine

{'answer': 'Locked accounts\nrequire IT Service Desk verification to unlock. All system passwords must meet these requirements:\n- Minimum 14 characters in length\n- Must contain uppercase, lowercase, numbers, and special characters\n- Passwords must be changed every 90 days\n- Previous 12 passwords cannot be reused\n- Multi-factor authentication (MFA) is mandatory for all systems\n- Passwords must not contain personal information or dictionary words\nAccount lockout occurs after 5 failed login attempts. [IT-001, SECTION 1: PASSWORD POLICY]\n\nLimited personal use\nis permitted if it does not interfere with work duties or violate any policy. Company IT resources are provided for business purposes. Prohibited activities include: unauthorized software installation, cryptocurrency\nmining, accessing inappropriate content, using company email for personal business,\nsharing credentials, and connecting unauthorized devices to the corporate network. [IT-001, SECTION 4: ACCEPTABLE USE]\n\nDat

In [21]:
# ── Filter by category ──
print("=== Query scoped to 'remote_work' category ===\n")
assistant.answer(
    "What are the communication expectations for employees?",
    metadata_filter={"category": "remote_work"}
)

=== Query scoped to 'remote_work' category ===

QUERY: What are the communication expectations for employees?
FILTER: category=remote_work

ANSWER:
  Remote employees must follow these communication standards: - Daily
  standup updates posted in team Slack channel by 10:30 AM - Weekly
  1-on-1 video call with direct manager - Monthly team all-hands
  meeting (in-person when possible) - Use project management tools for
  task tracking and updates - Document decisions and meeting notes in
  shared workspace - Escalate blockers within 4 hours of
  identification Asynchronous communication is preferred for non-
  urgent matters. [HR-002, SECTION 4: COMMUNICATION EXPECTATIONS]  The
  company provides the following for remote workers: - Laptop computer
  (standard company issue) - External monitor (up to 27 inches) -
  Keyboard and mouse - Headset for video calls - $500 one-time home
  office setup stipend Employees are responsible for their internet
  connection, desk, and chair. [HR-002, S

{'answer': 'Remote employees must follow these communication standards:\n- Daily standup updates posted in team Slack channel by 10:30 AM\n- Weekly 1-on-1 video call with direct manager\n- Monthly team all-hands meeting (in-person when possible)\n- Use project management tools for task tracking and updates\n- Document decisions and meeting notes in shared workspace\n- Escalate blockers within 4 hours of identification\nAsynchronous communication is preferred for non-urgent matters. [HR-002, SECTION 4: COMMUNICATION EXPECTATIONS]\n\nThe company provides the following for remote workers:\n- Laptop computer (standard company issue)\n- External monitor (up to 27 inches)\n- Keyboard and mouse\n- Headset for video calls\n- $500 one-time home office setup stipend\nEmployees are responsible for their internet connection, desk, and chair. [HR-002, SECTION 3: EQUIPMENT AND WORKSPACE]\n\nRemote work arrangements are available to employees who:\n- Have completed their 90-day probationary period\n-

In [22]:
# ── Compare filtered vs unfiltered ──
query = "What are the requirements for approval?"

print("=== UNFILTERED (all documents) ===\n")
unfiltered = assistant.answer(query, show_sources=False)

print("\n=== FILTERED: Finance only ===\n")
filtered = assistant.answer(query, metadata_filter={"department": "Finance"}, show_sources=False)

print("Comparison:")
print(f"  Unfiltered confidence: {unfiltered['confidence']['score']:.3f} ({unfiltered['confidence']['level']})")
print(f"  Filtered confidence:   {filtered['confidence']['score']:.3f} ({filtered['confidence']['level']})")
print(f"\n  Unfiltered sources: {[c['doc_id'] for c in unfiltered['citations']]}")
print(f"  Filtered sources:   {[c['doc_id'] for c in filtered['citations']]}")

=== UNFILTERED (all documents) ===


=== FILTERED: Finance only ===



Comparison:
  Unfiltered confidence: 0.515 (MEDIUM)
  Filtered confidence:   0.442 (LOW)

  Unfiltered sources: ['FIN-001', 'LEGAL-001', 'HR-002']
  Filtered sources:   ['FIN-001', 'FIN-001', 'FIN-001']


## Evaluation: Baseline vs RAG

We compare keyword search vs semantic retrieval across a set of test queries.
For each query, we check whether the correct source document appears in the top results.

In [23]:
# ── Evaluation: Baseline vs Semantic Retrieval ──

eval_queries = [
    {
        "query": "How many vacation days after 5 years?",
        "expected_doc": "HR-001",
        "expected_dept": "Human Resources",
    },
    {
        "query": "What is the minimum password length?",
        "expected_doc": "IT-001",
        "expected_dept": "IT Security",
    },
    {
        "query": "Maximum hotel rate per night?",
        "expected_doc": "FIN-001",
        "expected_dept": "Finance",
    },
    {
        "query": "How long are employee records retained?",
        "expected_doc": "LEGAL-001",
        "expected_dept": "Legal",
    },
    {
        "query": "Remote work internet speed requirements?",
        "expected_doc": "HR-002",
        "expected_dept": "Human Resources",
    },
    {
        "query": "What happens during a data breach?",
        "expected_doc": "LEGAL-001",
        "expected_dept": "Legal",
    },
    {
        "query": "Daily meal allowance for travel",
        "expected_doc": "FIN-001",
        "expected_dept": "Finance",
    },
    {
        "query": "Performance review schedule",
        "expected_doc": "HR-001",
        "expected_dept": "Human Resources",
    },
]

baseline_hits = 0
semantic_hits = 0
filtered_hits = 0

print(f"{'Query':<45} {'Baseline':^10} {'Semantic':^10} {'Filtered':^10}")
print("-" * 75)

for eq in eval_queries:
    query = eq["query"]
    expected = eq["expected_doc"]

    # Baseline
    b_results = keyword_search(query, all_chunks, top_k=3)
    b_hit = any(c.doc_id == expected for c, _ in b_results)
    baseline_hits += int(b_hit)

    # Semantic (unfiltered)
    s_results = vector_store.search(query, top_k=3)
    s_hit = any(c.doc_id == expected for c, _ in s_results)
    semantic_hits += int(s_hit)

    # Semantic (filtered by department)
    f_results = vector_store.search(query, top_k=3, metadata_filter={"department": eq["expected_dept"]})
    f_hit = any(c.doc_id == expected for c, _ in f_results)
    filtered_hits += int(f_hit)

    b_str = "HIT" if b_hit else "MISS"
    s_str = "HIT" if s_hit else "MISS"
    f_str = "HIT" if f_hit else "MISS"
    print(f"  {query:<43} {b_str:^10} {s_str:^10} {f_str:^10}")

n = len(eval_queries)
print("-" * 75)
print(f"  {'ACCURACY':<43} {baseline_hits/n:^10.1%} {semantic_hits/n:^10.1%} {filtered_hits/n:^10.1%}")
print(f"\nBaseline: {baseline_hits}/{n}, Semantic: {semantic_hits}/{n}, Filtered: {filtered_hits}/{n}")

Query                                          Baseline   Semantic   Filtered 
---------------------------------------------------------------------------


  How many vacation days after 5 years?          HIT        HIT        HIT    


  What is the minimum password length?           HIT        HIT        HIT    


  Maximum hotel rate per night?                  HIT        HIT        HIT    


  How long are employee records retained?        HIT        HIT        HIT    


  Remote work internet speed requirements?       HIT        HIT        HIT    


  What happens during a data breach?             MISS       HIT        HIT    


  Daily meal allowance for travel                HIT        HIT        HIT    


  Performance review schedule                    HIT        HIT        HIT    
---------------------------------------------------------------------------
  ACCURACY                                      87.5%      100.0%     100.0%  

Baseline: 7/8, Semantic: 8/8, Filtered: 8/8


## Error Analysis

Let's examine cases where retrieval might struggle and understand why.

In [24]:
# ── Error analysis: ambiguous and cross-document queries ──

edge_queries = [
    "What approvals do I need?",              # Ambiguous — spans multiple docs
    "Tell me about company benefits",          # Very broad
    "GDPR rights",                             # Jargon/abbreviation
    "What should I do in an emergency?",       # Could be IT or general
]

print("=== Edge Case Analysis ===\n")
for query in edge_queries:
    results = vector_store.search(query, top_k=3)
    confidence = compute_confidence(results)
    docs = [c.doc_id for c, _ in results]
    top_score = results[0][1] if results else 0.0

    print(f"Query: \"{query}\"")
    print(f"  Top score: {top_score:.3f} | Confidence: {confidence['score']:.3f} ({confidence['level']})")
    print(f"  Sources: {docs}")
    if confidence["level"] == "LOW":
        print(f"  WARNING: Low confidence — answer may be unreliable")
    print()

=== Edge Case Analysis ===

Query: "What approvals do I need?"
  Top score: 0.392 | Confidence: 0.558 (MEDIUM)
  Sources: ['FIN-001', 'LEGAL-001', 'FIN-001']



Query: "Tell me about company benefits"
  Top score: 0.532 | Confidence: 0.683 (MEDIUM)
  Sources: ['HR-001', 'IT-001', 'HR-002']

Query: "GDPR rights"
  Top score: 0.508 | Confidence: 0.655 (MEDIUM)
  Sources: ['LEGAL-001', 'LEGAL-001', 'IT-001']

Query: "What should I do in an emergency?"
  Top score: 0.218 | Confidence: 0.463 (LOW)
  Sources: ['IT-001', 'IT-001', 'LEGAL-001']



In [25]:
# ── Export metrics ──

metrics = {
    "project": "Multi-Document Policy Assistant",
    "corpus": {
        "documents": len(documents),
        "total_chunks": len(all_chunks),
        "departments": list(set(d.department for d in documents)),
        "categories": list(set(d.category for d in documents)),
    },
    "retrieval_backend": vector_store.backend,
    "evaluation": {
        "total_queries": len(eval_queries),
        "baseline_accuracy": baseline_hits / len(eval_queries),
        "semantic_accuracy": semantic_hits / len(eval_queries),
        "filtered_accuracy": filtered_hits / len(eval_queries),
    },
    "config": {
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "top_k": TOP_K,
        "embedding_model": EMBEDDING_MODEL if USE_SENTENCE_TRANSFORMERS else "TF-IDF",
    },
}

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\GenAI\11_Multi_Document_Policy_Assistant\metrics.json
{
  "project": "Multi-Document Policy Assistant",
  "corpus": {
    "documents": 5,
    "total_chunks": 29,
    "departments": [
      "Finance",
      "Legal",
      "Human Resources",
      "IT Security"
    ],
    "categories": [
      "expense_management",
      "remote_work",
      "data_privacy",
      "employee_benefits",
      "access_control"
    ]
  },
  "retrieval_backend": "chroma",
  "evaluation": {
    "total_queries": 8,
    "baseline_accuracy": 0.875,
    "semantic_accuracy": 1.0,
    "filtered_accuracy": 1.0
  },
  "config": {
    "chunk_size": 500,
    "chunk_overlap": 100,
    "top_k": 5,
    "embedding_model": "all-MiniLM-L6-v2"
  }
}


## Limitations

1. **No LLM generation** — answers are extractive (pulled from chunks), not generative.
   A production system would use an LLM (e.g., GPT-4, Llama 3) to synthesize
   natural-language answers from retrieved context.

2. **Static documents** — policies are hardcoded. A real system would watch a
   document repository and re-index on changes.

3. **No cross-reference resolution** — if Policy A says "see Policy B section 3",
   the system doesn't follow that reference.

4. **Simple confidence model** — the 4-factor confidence score is heuristic.
   Production systems would use calibrated models trained on relevance judgments.

5. **No access control** — in a real system, some policies might be restricted
   to certain employee groups. Metadata filtering would need to include
   permission checks.

6. **Chunk boundary issues** — important information may be split across chunks.
   Hierarchical chunking or parent-child retrieval would help.

## Common Mistakes in Multi-Document RAG

1. **Losing metadata during chunking** — If you chunk text without preserving metadata,
   you can't cite sources or filter by department. Always attach metadata to chunks
   before indexing.

2. **No metadata filtering** — Searching all documents for every query wastes compute
   and returns irrelevant results. Pre-filtering by known metadata drastically
   improves precision.

3. **Ignoring retrieval confidence** — Not all retrievals are equal. A low-similarity
   top chunk means the system is guessing. Always compute and surface confidence.

4. **Too-small chunks** — Very small chunks lose context. A chunk that says
   "5 days per year" without saying *what* those 5 days are for is useless.

5. **Too-large chunks** — Very large chunks dilute relevance. If a 2000-character
   chunk has one relevant sentence, the embedding averages out the relevance.

6. **Not comparing against a baseline** — If your fancy RAG system doesn't beat
   keyword search, something is wrong with your embedding or chunking strategy.

7. **Hardcoding document structure** — Assuming all documents have the same format
   breaks when you add new policy types. Parse documents generically.

## Mini Challenge

Try these exercises to deepen your understanding:

### Exercise 1: Adjustable Chunking
Modify `CHUNK_SIZE` and `CHUNK_OVERLAP` and re-run the evaluation.
- Try `CHUNK_SIZE=250, CHUNK_OVERLAP=50`
- Try `CHUNK_SIZE=1000, CHUNK_OVERLAP=200`
- Which setting gives the best accuracy?

### Exercise 2: Multi-Filter Queries
Write a query using multiple metadata filters:
```python
assistant.answer(
    "What is the leave policy?",
    metadata_filter={"department": "Human Resources", "category": "employee_benefits"}
)
```

### Exercise 3: Add a New Policy
Add a 6th policy document (e.g., "Health and Safety Policy") to `POLICY_DOCUMENTS`,
re-run the chunking and indexing, and test queries against it.

### Exercise 4: Improve Confidence Scoring
Modify `compute_confidence()` to add a 5th factor: query length.
Short queries ("vacation?") are inherently less precise than long queries.

### Exercise 5: Cross-Document Questions
Write a query that requires information from multiple documents:
*"What security measures apply to remote workers handling confidential data?"*
Does the system retrieve from both IT Security and Remote Work policies?

## Production Considerations

To turn this into a production system, you would need:

1. **LLM Integration** — Use an LLM (GPT-4, Claude, Llama 3) to generate natural
   answers from retrieved context instead of extractive assembly.

2. **Document Ingestion Pipeline** — Watch a blob store / SharePoint / Google Drive
   for policy changes, re-chunk and re-index automatically.

3. **Access Control** — Filter results based on the user's role and permissions.
   An HR intern should not see executive compensation policies.

4. **Versioning** — Track policy versions over time. When a user asks "what changed
   in the travel policy?", retrieve diffs between versions.

5. **Feedback Loop** — Let users rate answers. Use thumbs-up/down data to fine-tune
   retrieval and re-rank models.

6. **Hybrid Search** — Combine dense embeddings with sparse BM25/keyword search
   for better recall on exact-match queries (e.g., policy numbers).

7. **Scalability** — Use a managed vector database (Pinecone, Weaviate, Qdrant)
   for large document corpora with millions of chunks.

8. **Monitoring** — Track retrieval latency, confidence distributions, fallback rates,
   and user satisfaction metrics in production.

## How to Improve This Project

- **Add Reranking** — Use a cross-encoder model (e.g., `cross-encoder/ms-marco-MiniLM-L-6-v2`)
  to rerank retrieved chunks after initial retrieval.
- **Hierarchical Chunking** — Create parent-child relationships between chunks.
  Retrieve child chunks but expand to parent for more context.
- **Query Expansion** — Use an LLM to rephrase the query into multiple forms
  before retrieval, improving recall.
- **Semantic Caching** — Cache frequent query-answer pairs to reduce latency.
- **Document Summarization** — Generate per-document summaries for routing queries
  to the right document before chunk-level retrieval.
- **Evaluation Framework** — Build a labeled test set with ground-truth answers
  and compute NDCG, MAP, and MRR metrics.

## Key Takeaways

1. **Multi-document RAG requires metadata tracking** — every chunk must carry its
   source document, section, department, and temporal metadata.

2. **Metadata filtering is a precision multiplier** — filtering before retrieval
   narrows the search space and eliminates irrelevant results.

3. **Citations build trust** — showing users exactly where an answer came from
   lets them verify and builds confidence in the system.

4. **Confidence scoring prevents hallucination** — when retrieval quality is low,
   the system should say "I'm not sure" rather than guessing.

5. **Baselines matter** — always compare your RAG pipeline against simple keyword
   search. If semantic search doesn't beat keywords, something is misconfigured.

6. **Chunk size is a key hyperparameter** — too small = lost context, too large =
   diluted relevance. Experiment with different sizes for your corpus.